<div align="center">

# Universidad Autónoma de Chihuahua
## Facultad de Ingeniería
### Maestría en Ingeniería en Computación

**Materia:** Machine Learning (MIC1)

**Actividad:** Proyecto Final — Notebook 3: Modelos y optimización de hiperparámetros

**Alumno:** Jonahtan Gabriel González Cardoza
**Matrícula:** 345491

**Catedrática:** Dra. Olanda Prieto Ordaz

**Fecha de elaboración:** 21 de noviembre de 2026

</div>

**Proyecto:** predicción de la situación final de un estudiante (*Dropout*, *Enrolled*, *Graduate*) con el dataset
*Predict Students' Dropout and Academic Success* (UCI 697). Este proyecto reúne, en cinco notebooks reproducibles, el trabajo
de las tres prácticas anteriores (regresión logística del Proyecto F_1, SVM y métodos de ensamble).

**Objetivo de este notebook.** Evaluar con **validación cruzada de 5 folds sobre Train** (mismos folds para todos los modelos): modelos de referencia (regresión logística y SVM), ensambles con hiperparámetros base, y los mismos ensambles tras optimizar sus hiperparámetros. **Test no se carga en este notebook.** Al final guarda los modelos, las predicciones *out-of-fold* y las búsquedas para el notebook 04.

# III. Desarrollo (continuación)

In [ ]:
from config import *

# Test NO se carga en este notebook: la selección de modelos usa solo Train
X_train, _, y_train, _ = cargar_particion()
numeric_cols, nominal_cols, binary_cols = columnas()
reg = Registro(X_train, y_train)

# Alias: el resto del notebook usa los mismos nombres que la práctica de ensambles
resultados_cv, oof, modelos_ajustados = reg.resultados_cv, reg.oof, reg.modelos_ajustados
searches, tiempos_busqueda = reg.searches, reg.tiempos_busqueda
evaluar_cv, tabla_resumen, tabla_por_clase = reg.evaluar_cv, reg.tabla_resumen, reg.tabla_por_clase
comparar_oob, oob_res = reg.comparar_oob, reg.oob_res
buscar = reg.buscar
print(f"Train: {X_train.shape[0]:,} estudiantes × {X_train.shape[1]} características")

### 8. ¿Hace falta una submuestra de Train?
Con un dataset de cientos de miles de instancias, los modelos de costo cuadrático o cúbico (como un `SVC` con kernel) solo pueden optimizarse sobre una **submuestra estratificada**. Aquí Train tiene 3,539 estudiantes, y el costo de los ensambles crece aproximadamente de forma lineal con el número de estudiantes y con el de árboles. Se mide en las secciones siguientes (columna "Ajuste con todo Train (s)"): cada modelo tarda de menos de un segundo a ≈1 min en ajustarse (el *Stacking*, que entrena sus modelos base una vez por fold interno, es el más lento).

**No se submuestrea:** la validación cruzada, las búsquedas y los modelos finales usan todo Train, igual que la regresión logística del Proyecto F_1 y que el SVM de la práctica anterior. Con 635 *Enrolled* en Train, una submuestra solo quitaría información justo de la clase difícil. Test se usa completo y una sola vez, al final.

## Fase 5. Experimentación con ensambles

### 9. Configuración común
Todos los modelos se evalúan con el **mismo** `StratifiedKFold(n_splits=5, shuffle=True, random_state=42)` del Proyecto F_1, así que los folds son idénticos y los números de la regresión logística y del SVM se reproducen. Para cada modelo se registran media ± desviación de cada métrica, el score en los folds de entrenamiento (para medir el *gap*), las predicciones *out-of-fold* (OOF), el número de estimadores y los tiempos.

**Se reportan todas las métricas, no solo las favorables.** Varios trabajos publicados con este dataset reportan una sola métrica, sin el protocolo de partición o solo por clase (ver Fase 11), lo que impide saber si un resultado es bueno o solo está bien presentado. Aquí se reporta siempre el conjunto completo:

| Métrica | Qué mide con 3 clases |
|---|---|
| Accuracy | Proporción de estudiantes en su clase correcta. Engañosa con desbalance (el Dummy logra 0.4993) |
| Precision, Recall, F1 (**macro**) | Promedio sin ponderar de la métrica de cada clase; *Enrolled* pesa igual que *Graduate*. El Recall macro es la *balanced accuracy* |
| F1 **ponderado** | Promedio de los F1 por clase ponderado por su frecuencia (el "F1 global" de Martins et al., 2023). Favorece a la clase mayoritaria |
| MCC (Matthews) | Correlación entre clase real y predicha usando toda la matriz de confusión; 0 = azar, 1 = perfecto. Es robusto al desbalance |
| Kappa de Cohen | Acuerdo entre real y predicha descontando el acuerdo esperado por azar |
| ROC-AUC (**OvR macro**) | Un AUC por clase (esa clase contra el resto), promediado. Mide el **ordenamiento**, sin depender de la regla de decisión |
| Por clase | Precision, Recall, F1 y AUC de cada clase (OOF) |

> **ROC-AUC según el modelo.** Los ensambles con probabilidades (todos menos el *Voting hard*) usan `predict_proba`. El *Voting hard* vota por clase y no produce un puntaje continuo, así que su ROC-AUC no existe ("—" en las tablas). `SVC` solo ofrece `decision_function`, que con 3 clases resume votos uno contra uno; esa diferencia se analiza en la Sección 26. Dentro del *Voting soft* y del *Stacking* el SVM se calibra con `CalibratedClassifierCV` (sigmoide), que sí da probabilidades.

In [ ]:
# El CV (StratifiedKFold de 5 folds, shuffle, random_state=42), las 8 métricas, evaluar_cv(), tabla_resumen(), etc. están en config.py
print("Validación cruzada:", cv)
print("Métricas registradas:", METRICAS)
print(f"Train: {X_train.shape[0]:,} estudiantes (Test no se carga en este notebook)")

### 9.1 Acumulado de los mejores modelos (referencias)
De las prácticas anteriores se conservan, **sin repetir sus entrenamientos ni sus búsquedas**, solo los mejores modelos, con sus hiperparámetros fijos:

- `DummyClassifier(strategy="most_frequent")`: piso mínimo (siempre predice *Graduate*).
- **Regresión logística del Proyecto F_1**: `C = 0.3`, penalización L1 (solver `saga`) y `class_weight = "balanced"`. Su regla de decisión oficial es *Dropout* si P(*Dropout*) ≥ 0.40 (se aplica solo en Test, Sección 24; en CV se usa la regla por defecto, la clase de mayor probabilidad).
- **El mejor SVM de cada kernel** (práctica de SVM): lineal (`LinearSVC`, `C = 0.01`, `class_weight = "balanced"`), polinomial (`degree = 2`, `coef0 = 1`, `C = 1`, `class_weight = "balanced"`) y RBF (`C = 100`, `gamma = 0.001`, sin pesos; el SVM final de esa práctica).

Se descartan los SVM con parámetros por defecto y las mallas de búsqueda de los SVM. Los modelos de referencia se vuelven a evaluar aquí con exactamente los mismos folds, para compararlos con los ensambles con la misma vara.

In [ ]:
# BEST_LR, BEST_SVM, NOMBRE_LR y predecir_regla() están en config.py (hiperparámetros fijos; NO se repiten sus búsquedas)

modelos_ref = {
    "Dummy (referencia)": Pipeline([("model", DummyClassifier(strategy="most_frequent"))]),
    NOMBRE_LR: crear_pipeline(LogisticRegression(**BEST_LR, max_iter=5000, random_state=RANDOM_STATE)),
    **{n: crear_pipeline(d["modelo"](**d["params"])) for n, d in BEST_SVM.items()},
}
for nombre, pipe in modelos_ref.items():
    evaluar_cv(nombre, pipe)

# Modelos BASE de referencia: LR y SVM con los hiperparámetros por defecto de scikit-learn (para la comparación base vs. optimizado)
modelos_defecto = {
    "Regresión Logística base (por defecto)": crear_pipeline(LogisticRegression(max_iter=5000, random_state=RANDOM_STATE)),
    "SVM Lineal (por defecto)": crear_pipeline(LinearSVC(max_iter=10_000, random_state=RANDOM_STATE)),
    "SVM Polinomial (por defecto)": crear_pipeline(SVC(kernel="poly", cache_size=1000)),
    "SVM RBF (por defecto)": crear_pipeline(SVC(kernel="rbf", cache_size=1000)),
}
for nombre, pipe in modelos_defecto.items():
    evaluar_cv(nombre, pipe)

In [ ]:
display(tabla_resumen())
print("Recall por clase (OOF):")
display(tabla_por_clase("Recall").round(4))
print("F1 por clase (OOF):")
display(tabla_por_clase("F1").round(4))
guardar_tabla(tabla_resumen(), "03_referencias_cv")

**Interpretación.**
- **Los modelos de referencia reproducen exactamente sus resultados anteriores** (mismos folds): regresión logística F1 macro 0.7170 ± 0.0192 y ROC-AUC 0.8913 ± 0.0104 (Proyecto F_1); SVM lineal 0.7086, polinomial 0.7211 y RBF 0.7217 (práctica de SVM). Aquí se reportan además F1 ponderado, MCC y Kappa: para la regresión logística 0.7663, 0.6178 y 0.6146.
- **El Dummy marca el piso:** Accuracy 0.4993, F1 macro 0.2220, MCC y Kappa iguales a 0 (no es mejor que el azar) y ROC-AUC 0.5. Su F1 ponderado (0.3325) es mayor que su F1 macro: premia acertar la clase mayoritaria, por eso no basta para evaluar este problema.
- **Las referencias están en un mismo rango de F1 macro (0.7086–0.7217)**, pero tienen perfiles distintos. El SVM RBF tiene el mayor Accuracy (0.7872) y MCC (0.6483), y reconoce a 93.0% de los *Graduate* pero a solo 43.6% de los *Enrolled* (Recall OOF); la regresión logística y el SVM polinomial, que están ponderados, reconocen a 61.1% y 62.1% de los *Enrolled* y a 82.6% y 82.9% de los *Graduate*.
- **El gap train − CV es pequeño** en la regresión logística (0.033), el SVM lineal (0.024) y el RBF (0.043); el polinomial sobreajusta más (0.103).
- El ROC-AUC del SVM (0.8630 y 0.8742 en los otros kernels) se calcula con `decision_function` y no es comparable con el de los modelos con probabilidades (Sección 26).

### 9.2 Regla de decisión de la regresión logística (Proyecto F_1)
La regresión logística se evalúa con dos reglas: la clase más probable (por defecto) y la regla de alerta temprana *Dropout si P(Dropout) ≥ t*. El threshold `t`
se elige aquí con las predicciones *out-of-fold* de Train, con el criterio declarado de antemano en el Proyecto F_1 (mayor F1 macro; si varias quedan a menos de
0.005, la de mayor Recall de *Dropout*). Test no interviene en esta decisión.

In [ ]:
# Regla de decisión de la regresión logística (Proyecto F_1): se elige con las predicciones OOF de Train, NO con Test
score_lr = oof[NOMBRE_LR][1]                    # probabilidades OOF de las 3 clases
filas = []
for t in [None, 0.30, 0.40, 0.50, 0.60, 0.70]:
    y_pred_t = predecir_regla(score_lr, t)
    es_d, pred_d = (y_train == 0), (y_pred_t == 0)
    filas.append({"Regla": "Clase más probable" if t is None else f"P(Dropout) ≥ {t:.2f}", "umbral": t,
                  "Precision (Dropout)": precision_score(es_d, pred_d), "Recall (Dropout)": recall_score(es_d, pred_d),
                  "F1 (Dropout)": f1_score(es_d, pred_d), "Recall (Enrolled)": recall_score(y_train == 1, y_pred_t == 1),
                  "F1 macro": f1_score(y_train, y_pred_t, average="macro"), "Accuracy": accuracy_score(y_train, y_pred_t),
                  "Falsos positivos (Dropout)": int((~es_d & pred_d).sum()), "Falsos negativos (Dropout)": int((es_d & ~pred_d).sum())})
tabla_threshold = pd.DataFrame(filas).set_index("Regla")
display(tabla_threshold.drop(columns="umbral").round(4))

# Criterio declarado antes de usar Test: mayor F1 macro; si varias quedan a menos de 0.005, la de mayor Recall de Dropout
f1_max = tabla_threshold["F1 macro"].max()
candidatas = tabla_threshold[tabla_threshold["F1 macro"] >= f1_max - 0.005]
regla_final = candidatas["Recall (Dropout)"].idxmax()
UMBRAL_LR = tabla_threshold.loc[regla_final, "umbral"]
UMBRAL_LR = None if pd.isna(UMBRAL_LR) else float(UMBRAL_LR)
reg.extra["umbral_lr"] = UMBRAL_LR
print("Regla seleccionada con OOF de Train:", regla_final, "| en el Proyecto F_1 se había fijado P(Dropout) ≥ 0.40")
assert UMBRAL_LR == 0.40, "la regla elegida no coincide con la del Proyecto F_1 (revisa versiones de librerías)"

tabla_t = tabla_threshold.dropna(subset=["umbral"]).set_index("umbral")
fig, ax = plt.subplots(figsize=(8.5, 4.8))
for metrica, color in [("Precision (Dropout)", "#2a78d6"), ("Recall (Dropout)", "#eb6834"),
                       ("F1 (Dropout)", "#1baf7a"), ("F1 macro", "#eda100")]:
    ax.plot(tabla_t.index, tabla_t[metrica], marker="o", lw=2, color=color, label=metrica)
    ax.axhline(tabla_threshold.loc["Clase más probable", metrica], color=color, lw=1, ls=":")
ax.axvline(UMBRAL_LR, color=COLOR_GRIS, lw=1)
ax.set_xlabel("Threshold sobre P(Dropout)")
ax.set_ylabel("Valor (OOF, Train)")
ax.set_title("Regresión logística — métricas por threshold (punteado: clase más probable)")
ax.legend(loc="upper left", bbox_to_anchor=(1.01, 1.0))
plt.tight_layout()
guardar_fig("03_threshold_lr")
guardar_tabla(tabla_threshold, "03_threshold_lr")

### 10. Voting Classifiers
Un *voting classifier* combina modelos **de naturaleza distinta** (que se equivocan de forma distinta) y decide por votación. Se usan tres miembros: la **regresión logística** del Proyecto F_1 (lineal), el **SVM RBF** optimizado (basado en distancias) y un **Random Forest** (árboles).

- **Hard voting:** cada modelo vota una clase y gana la más votada.
- **Soft voting:** se promedian las probabilidades de cada clase y gana la de mayor promedio; usa más información (la confianza de cada modelo) pero exige probabilidades. El SVM se envuelve en `CalibratedClassifierCV(method="sigmoid", cv=5)`.

La condición para que el *voting* mejore es que los errores de los miembros no estén muy correlacionados.

In [ ]:
def modelos_base(probabilidades=False):
    """Los tres modelos base del Voting y del Stacking: LR (F_1), SVM RBF optimizado y Random Forest."""
    lr = LogisticRegression(**BEST_LR, max_iter=5000, random_state=RANDOM_STATE)
    svm = SVC(**BEST_SVM["SVM RBF optimizado"]["params"])
    if probabilidades:   # el soft voting necesita probabilidades: se calibra el SVM (sigmoide / Platt)
        svm = CalibratedClassifierCV(svm, method="sigmoid", cv=5, ensemble=False)
    rf = RandomForestClassifier(n_estimators=200, min_samples_leaf=5, class_weight="balanced_subsample",
                                n_jobs=1, random_state=RANDOM_STATE)
    return [("lr", lr), ("svm", svm), ("rf", rf)]

voting_hard = crear_pipeline(VotingClassifier(modelos_base(False), voting="hard"))
voting_soft = crear_pipeline(VotingClassifier(modelos_base(True), voting="soft"))
evaluar_cv("Voting (hard)", voting_hard)
evaluar_cv("Voting (soft)", voting_soft);

**Interpretación.**
- **Hard voting:** F1 macro 0.7289 ± 0.0175, Accuracy 0.7796, MCC 0.6401. Supera por +0.012 de F1 macro a la regresión logística (0.7170), una diferencia menor que una desviación entre folds. No tiene ROC-AUC (vota clases, no produce puntaje).
- **Soft voting:** F1 macro 0.7258 ± 0.0176, pero el mayor Accuracy (0.7898) y MCC (0.6525) de los ensambles base, y un ROC-AUC de 0.8966, mayor que el de la regresión logística (0.8913) y el del SVM con `decision_function` (0.8630). Promediar probabilidades ordena mejor a los estudiantes que cualquiera de sus tres miembros por separado.
- **El soft voting hereda el sesgo del SVM sin pesos:** reconoce a solo 44.7% de los *Enrolled* (OOF) y a 92.2% de los *Graduate*, contra 53.7% y 87.9% del *hard voting*. Es el mismo compromiso Accuracy contra Recall de *Enrolled* que se vio con los SVM.
- **Gap:** 0.077 (hard) y 0.052 (soft), moderado: el *voting* no entrena nada nuevo, solo combina modelos ya regularizados (el RF es el que aporta el gap).

### 11. Bagging y Pasting
Ambos entrenan muchos árboles, cada uno con una submuestra aleatoria de Train, y promedian sus predicciones, lo que **reduce la varianza**. **Bagging** muestrea *con* reemplazo (`bootstrap=True`); **Pasting**, *sin* reemplazo (`bootstrap=False`). El árbol base está restringido (`max_depth = 8`, `min_samples_leaf = 10`, `class_weight = "balanced"`), y cada ensamble usa 200 árboles con `max_samples = 0.5`. Como referencia se evalúa un **árbol único** con la misma configuración: la diferencia contra el ensamble es el efecto directo de promediar.

In [ ]:
arbol_base = DecisionTreeClassifier(max_depth=8, min_samples_leaf=10, class_weight="balanced", random_state=RANDOM_STATE)
evaluar_cv("Árbol de decisión (1 árbol)", crear_pipeline(clone(arbol_base)))
bagging = crear_pipeline(BaggingClassifier(estimator=arbol_base, n_estimators=200, max_samples=0.5, bootstrap=True,
                                           n_jobs=1, random_state=RANDOM_STATE))
pasting = crear_pipeline(BaggingClassifier(estimator=arbol_base, n_estimators=200, max_samples=0.5, bootstrap=False,
                                           n_jobs=1, random_state=RANDOM_STATE))
evaluar_cv("Bagging (árboles)", bagging)
evaluar_cv("Pasting (árboles)", pasting);

### 11.1 Evaluación Out-of-Bag (OOB)
Con `bootstrap=True`, cada árbol deja fuera ≈37% de los estudiantes; con 200 árboles, cada estudiante queda fuera de decenas de ellos, y se puede predecir usando **solo los árboles que no lo vieron**. Eso da una estimación de generalización **sin validación cruzada ni conjunto de validación**. Se calcula con `oob_decision_function_` (las probabilidades OOB) y con ellas se obtienen las mismas ocho métricas; el `oob_score_` de scikit-learn es solo Accuracy, poco informativa con clases 50/32/18. Se hace también para el Random Forest de la sección siguiente.

In [ ]:
comparar_oob("Bagging (OOB)", bagging, "Bagging (árboles)")

**Interpretación.**
- **Promediar árboles funciona:** el árbol único alcanza F1 macro 0.6559 (Accuracy 0.6917, ROC-AUC 0.8362) y el *bagging* de 200 árboles con la misma configuración sube a 0.7023 (+0.046; Accuracy 0.7465, ROC-AUC 0.8759); el *pasting*, a 0.7004. El efecto es reducción de varianza: el árbol aislado es inestable y el promedio no.
- **Bagging y pasting son equivalentes aquí** (diferencia de 0.002 de F1 macro, mucho menor que la desviación entre folds, ≈0.013–0.016). Con `max_samples = 0.5` los dos muestrean subconjuntos muy distintos entre árboles, así que reemplazar o no reemplazar importa poco.
- **Sin sobreajuste notable:** gap de 0.052 (bagging) y 0.065 (pasting), frente a 0.087 del árbol único.
- **OOB:** las 3,539 instancias quedaron fuera de al menos un árbol. Las métricas OOB reproducen casi exactamente las de la validación cruzada: F1 macro 0.7039 contra 0.7023, Accuracy 0.7460 contra 0.7465, MCC 0.5966 contra 0.5962 y ROC-AUC 0.8765 contra 0.8759. **Con un solo ajuste se obtiene la misma estimación que con 5 ajustes**, sin separar datos de validación. Por clase (OOB), el *bagging* reconoce a 69.4% de los *Dropout*, 60.0% de los *Enrolled* y 83.2% de los *Graduate*.

### 12. Random Forest
Un Random Forest es *bagging* de árboles más un segundo nivel de aleatoriedad: en cada división solo se considera un subconjunto aleatorio de variables (`max_features = "sqrt"`), lo que **decorrelaciona** los árboles y reduce más la varianza del promedio. `class_weight = "balanced_subsample"` recalcula los pesos de clase en la muestra *bootstrap* de cada árbol y `min_samples_leaf = 5` evita árboles que memorizan.

In [ ]:
random_forest = crear_pipeline(RandomForestClassifier(n_estimators=300, max_features="sqrt", min_samples_leaf=5,
                                                      class_weight="balanced_subsample", n_jobs=1,
                                                      random_state=RANDOM_STATE))
evaluar_cv("Random Forest", random_forest);

### 12.1 Evaluación Out-of-Bag del Random Forest
Mismo procedimiento que en la Sección 11.1: el Random Forest también entrena cada árbol con un *bootstrap*, así que cada estudiante tiene predicciones OOB.

In [ ]:
comparar_oob("Random Forest (OOB)", random_forest, "Random Forest")

**Interpretación.**
- **Random Forest:** F1 macro 0.7176 ± 0.0220, Accuracy 0.7657, MCC 0.6198 y ROC-AUC 0.8863. Mejora +0.015 de F1 macro y +0.010 de ROC-AUC al *bagging* (decorrelacionar los árboles con `max_features = "sqrt"` ayuda), y queda al nivel de la regresión logística (0.7170).
- **Es el ensamble base que más sobreajusta de los dos de árboles independientes:** gap 0.146 (los árboles crecen hasta hojas de 5 estudiantes). Eso no lo hace peor en CV: el sobreajuste de cada árbol lo cancela el promedio, pero sí indica que hay margen para regularizar (Fase 6).
- **OOB:** F1 macro 0.7158 contra 0.7176 de la CV, Accuracy 0.7652 contra 0.7657, ROC-AUC 0.8874 contra 0.8863. Igual que en el *bagging*, la estimación sin validación coincide con la de 5 folds. Por clase (OOB): 73.3% de los *Dropout*, 54.3% de los *Enrolled* y 86.6% de los *Graduate*.

### 13. Boosting: AdaBoost, Gradient Boosting e HistGradientBoosting
A diferencia del *bagging*, el *boosting* entrena modelos débiles **en secuencia**, cada uno corrigiendo los errores del anterior, y **reduce el sesgo**.

- **AdaBoost** (SAMME) re-pondera los estudiantes mal clasificados; base: árboles de profundidad 2 con `class_weight = "balanced"`, 200 estimadores, `learning_rate = 0.5`.
- **Gradient Boosting** ajusta cada árbol al gradiente de la pérdida (multinomial: un árbol por clase en cada iteración). `GradientBoostingClassifier` no tiene `class_weight`, así que una subclase calcula pesos `balanced` por muestra dentro de `fit()` (también dentro de cada fold). 200 árboles de profundidad 3, `learning_rate = 0.1`, `subsample = 0.8`.
- **HistGradientBoosting** discretiza las variables en *bins* (mucho más rápido); mismo algoritmo con `class_weight = "balanced"`, 200 iteraciones, 15 hojas máximo por árbol.

In [ ]:
# GradientBoostingBalanceado está en config.py (pesos 'balanced' calculados dentro de fit())
adaboost = crear_pipeline(AdaBoostClassifier(
    estimator=DecisionTreeClassifier(max_depth=2, class_weight="balanced", random_state=RANDOM_STATE),
    n_estimators=200, learning_rate=0.5, random_state=RANDOM_STATE))
gboost = crear_pipeline(GradientBoostingBalanceado(
    n_estimators=200, learning_rate=0.1, max_depth=3, subsample=0.8, random_state=RANDOM_STATE))
hgb = crear_pipeline(HistGradientBoostingClassifier(
    max_iter=200, learning_rate=0.1, max_leaf_nodes=15, class_weight="balanced", early_stopping=False,
    random_state=RANDOM_STATE))

evaluar_cv("AdaBoost", adaboost)
evaluar_cv("Gradient Boosting (balanceado)", gboost)
evaluar_cv("HistGradientBoosting (balanceado)", hgb);

**Interpretación.**
- **Gradient Boosting (balanceado):** el mejor de los ensambles base por F1 macro (0.7300 ± 0.0214), con Accuracy 0.7714, MCC 0.6345 y ROC-AUC 0.8932; reconoce a 60.8% de los *Enrolled* (OOF). Gap 0.167: sobreajusta, pero menos que el HistGradientBoosting.
- **HistGradientBoosting:** F1 macro 0.7164 y el mayor gap (0.282: casi perfecto en entrenamiento). Con 15 hojas por árbol y sin regularización, memoriza a Train; la optimización de la Fase 6 se concentra en eso.
- **AdaBoost:** el peor de los ensambles base de *boosting* (F1 macro 0.6784, Accuracy 0.7268, ROC-AUC 0.8536) y con gap casi nulo (0.006). Solo construyó **10 de los 200 estimadores**: scikit-learn detiene el *boosting* cuando un estimador deja de ser mejor que el azar con los pesos actuales. AdaBoost es sensible a instancias difíciles o ruidosas porque les sube el peso en cada ronda; con una clase intermedia y ambigua como *Enrolled* es una explicación plausible (no se verificó). La optimización (Fase 6) explora otras profundidades y *learning rates*.

### 14. Stacked Generalization (Stacking)
En lugar de votar con una regla fija, el *stacking* **aprende cómo combinar** los modelos base: un meta-modelo (el *blender*) recibe como entradas las predicciones de los modelos base y aprende a ponderarlas. Para que el *blender* no se entrene con predicciones que los modelos base hicieron sobre sus propios datos de entrenamiento (optimistas), `StackingClassifier` usa **predicciones *out-of-fold*** de los modelos base (`cv` de 5 folds, la misma semilla). Modelos base: regresión logística, SVM RBF y Random Forest; *blender*: regresión logística con `class_weight = "balanced"`.

In [ ]:
stacking = crear_pipeline(StackingClassifier(
    estimators=modelos_base(False),
    final_estimator=LogisticRegression(class_weight="balanced", max_iter=2000, random_state=RANDOM_STATE),
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE),
    stack_method="auto", n_jobs=1))
evaluar_cv("Stacking (LR + SVM + RF → LR)", stacking);

**Interpretación.**
- **Stacking:** F1 macro 0.7217 ± 0.0207, Accuracy 0.7595, MCC 0.6228 y ROC-AUC 0.8960 (el segundo mayor de los ensambles base, después del *soft voting*, 0.8966).
- **El *blender* balanceado corrige el sesgo del SVM:** reconoce a 64.6% de los *Enrolled* (OOF), contra 44.7% del *soft voting* y 43.6% del SVM solo, con el mismo SVM y el mismo RF entre sus miembros. Aprender la combinación (con pesos de clase) rinde más que promediar con pesos fijos.
- **Pero en F1 macro no supera a sus miembros:** 0.7217 contra 0.7170 de la regresión logística, 0.7217 del SVM y 0.7176 del RF. Es **la combinación más cara**: ajustarlo con todo Train toma ≈25 s, contra menos de 7 s de los demás ensambles, porque entrena sus tres modelos base cinco veces más (una por fold interno) y luego una vez más.

### 15. Explainable Boosting Machine (EBM)
La EBM (Nori et al., 2019; paquete `interpret`) es un **modelo aditivo generalizado** entrenado con *boosting*: $\text{logit}(P) = \beta_0 + \sum_j f_j(x_j) + \sum_{j,k} f_{jk}(x_j, x_k)$. Cada $f_j$ se aprende dando pasos pequeños **de una variable a la vez** (en ciclos), así que el modelo final es una suma de funciones de forma que se pueden graficar: es un ensamble (de miles de árboles muy pequeños) que conserva la **interpretabilidad** de un modelo aditivo.

- Trabaja con las variables originales, sin escalar ni *one-hot*; se declaran las 9 nominales como `"nominal"`.
- `outer_bags = 8` (promedia 8 modelos entrenados con submuestras distintas; da las bandas de incertidumbre).
- Pesos de clase `balanced` mediante una subclase, igual que en Gradient Boosting.
- Se evalúan dos versiones: **sin interacciones** (GAM puro) y con **10 interacciones** de a pares. **Limitación:** con más de dos clases, `interpret` calcula las interacciones pero **no puede graficarlas**; por eso las funciones de forma de la Sección 29 se muestran con el GAM sin interacciones.

In [ ]:
# EBMBalanceado y crear_ebm() están en config.py (las 9 nominales se declaran como categóricas)
with warnings.catch_warnings():
    warnings.simplefilter("ignore", UserWarning)       # aviso de interpret: pares no visualizables en multiclase
    evaluar_cv("EBM sin interacciones (GAM)", crear_ebm(0))
    evaluar_cv("EBM (10 interacciones)", crear_ebm(10));

**Interpretación.**
- **EBM sin interacciones (GAM):** F1 macro 0.7143 ± 0.0183, Accuracy 0.7519, ROC-AUC 0.8855 y gap 0.043. Es prácticamente lo mismo que la regresión logística (0.7170): **aprender una función de forma no lineal por variable no aporta nada** por sí solo.
- **EBM con 10 interacciones:** F1 macro 0.7285 ± 0.0095 (la desviación entre folds es la mitad que la de los demás), MCC 0.6311, ROC-AUC 0.8923 y el mayor Recall de *Enrolled* de los ensambles base (64.7%). Las interacciones aportan +0.0142 de F1 macro. La mejora, si es real, viene de combinaciones de variables y no de curvas no lineales.
- **Costo:** 3 s (GAM) y 5 s (con interacciones) por ajuste con todo Train.

### 16. Tabla comparativa inicial (Cross-Validation, 5 folds)
Los ensambles con hiperparámetros **base** (razonables pero sin buscar), junto con las referencias. El SVM RBF de la tabla es el de la práctica anterior, ya optimizado.

In [ ]:
nombres_base = ["Dummy (referencia)", *modelos_defecto, NOMBRE_LR, "SVM RBF optimizado", "Árbol de decisión (1 árbol)", "Voting (hard)", "Voting (soft)",
                "Bagging (árboles)", "Pasting (árboles)", "Random Forest", "AdaBoost", "Gradient Boosting (balanceado)",
                "HistGradientBoosting (balanceado)", "Stacking (LR + SVM + RF → LR)", "EBM sin interacciones (GAM)", "EBM (10 interacciones)"]
display(tabla_resumen(nombres_base))
print("Recall por clase (OOF):")
display(tabla_por_clase("Recall", nombres_base).round(4))
print("F1 por clase (OOF):")
display(tabla_por_clase("F1", nombres_base).round(4))

orden = sorted(nombres_base[1:], key=lambda n: resultados_cv[n]["F1_mean"])
fig, ax = plt.subplots(figsize=(10, 6))
y_pos = np.arange(len(orden))
ax.barh(y_pos, [resultados_cv[n]["F1_mean"] for n in orden], xerr=[resultados_cv[n]["F1_std"] for n in orden], capsize=3,
        color=["tab:red" if n == NOMBRE_LR else "tab:blue" for n in orden], alpha=0.85)
ax.scatter([resultados_cv[n]["F1_train"] for n in orden], y_pos, color="black", s=18, zorder=3, label="F1 en los folds de entrenamiento")
ax.set_yticks(y_pos)
ax.set_yticklabels(orden)
ax.set_xlim(0.5, 1.02)
ax.set_xlabel("F1 macro (barra: CV, media ± std; punto: train)")
ax.set_title("Ensambles con hiperparámetros base — F1 macro (rojo = regresión logística)")
ax.legend(loc="lower right")
plt.tight_layout()
guardar_fig("03_modelos_base_f1_cv")
guardar_tabla(tabla_resumen(nombres_base), "03_modelos_base_cv")

**Interpretación.**
- **Por F1 macro (la métrica de selección)** el orden es: Gradient Boosting 0.7300, *Voting hard* 0.7289, EBM con interacciones 0.7285, *Voting soft* 0.7258, Stacking y SVM RBF 0.7217, Random Forest 0.7176, regresión logística 0.7170, HistGradientBoosting 0.7164, EBM sin interacciones 0.7143, Bagging 0.7023, Pasting 0.7004, AdaBoost 0.6784 y árbol único 0.6559. Salvo los últimos cuatro, **los modelos están en un rango de 0.016 de F1 macro, menor que una desviación entre folds (0.01–0.02)**: con 5 folds no se pueden distinguir.
- **Por Accuracy y MCC** gana el *Voting soft* (0.7898 y 0.6525), seguido del SVM RBF (0.7872 y 0.6483); **por ROC-AUC**, también el *Voting soft* (0.8966) y el *Stacking* (0.8960), por encima de la regresión logística (0.8913). Los modelos con más Accuracy son los que menos reconocen a *Enrolled* (Recall OOF 44.7% el *soft voting*, 43.6% el SVM), y los de más Recall de *Enrolled* (EBM con interacciones 64.7%, Stacking 64.6%, regresión logística 61.1%) pierden Accuracy.
- **Sobreajuste:** el gap train − CV crece con la capacidad: 0.006 (AdaBoost, que se detuvo en 10 estimadores), 0.033 (regresión logística), 0.043–0.057 (SVM y EBM), 0.146 (Random Forest), 0.167 (Gradient Boosting) y 0.282 (HistGradientBoosting). Un gap alto no implica un peor desempeño en CV, pero sí que hay margen para regularizar.

## Fase 6. Optimización de hiperparámetros

### 17. Qué se optimiza y cómo
Se optimizan los siete ensambles que tienen hiperparámetros propios (Random Forest, Bagging, Pasting, AdaBoost, Gradient Boosting, HistGradientBoosting y EBM), más los pesos del *voting* y el *blender* del *stacking*. La regresión logística y el SVM **no** se vuelven a optimizar (se conserva el acumulado). Todas las búsquedas usan los **mismos 5 folds** y la métrica de selección que se había declarado desde el Proyecto F_1: **F1 macro** (`refit = "F1"`); se registran también las otras siete.

| Modelo | Hiperparámetros explorados | Combinaciones | Búsqueda |
|---|---|---:|---|
| Random Forest | `max_features` {sqrt, log2, 0.2, 0.35, 0.5} × `min_samples_leaf` {1, 2, 3, 5, 8, 12, 20} × `max_depth` {None, 8, 12, 16} × `class_weight` {None, balanced, balanced_subsample} × `criterion` {gini, entropy} | 840 | aleatoria, 40 |
| Bagging | `max_samples` {0.3, 0.5, 0.7, 1.0} × `max_features` {0.3, 0.6, 1.0} × profundidad y `min_samples_leaf` del árbol base × `class_weight` | 480 | aleatoria, 30 |
| Pasting | igual, con `max_samples` {0.3, 0.5, 0.7, 0.9} | 480 | aleatoria, 30 |
| AdaBoost | `n_estimators` {25, 50, 100, 200, 400} × `learning_rate` {0.03, 0.1, 0.3, 1} × `max_depth` del árbol base {1, 2, 3, 4, 6, 8, 10} | 140 | exhaustiva |
| Gradient Boosting | `learning_rate` {0.02, 0.05, 0.1, 0.2} × `n_estimators` {100, 200, 300} × `max_depth` {2–5} × `subsample` {0.5, 0.7, 1} × `min_samples_leaf` {1, 5, 10, 20} × `max_features` {None, sqrt, 0.3} | 1,728 | aleatoria, 30 |
| HistGradientBoosting | `learning_rate` × `max_iter` × `max_leaf_nodes` × `min_samples_leaf` × `l2_regularization` × `max_depth` × `class_weight` | 2,592 | aleatoria, 40 |
| EBM | `interactions` {0, 10, 20} × `learning_rate` {0.01, 0.03} × `max_leaves` {2, 3} × `min_samples_leaf` {4, 20} | 24 | exhaustiva |
| Voting (hard y soft) | 5 vectores de pesos para (LR, SVM, RF, GB, HGB) | 5 | exhaustiva |
| Stacking | $C$ del *blender* ∈ {0.01, 0.1, 1, 10} con validación cruzada interna | 4 | `LogisticRegressionCV` |

Cuando la malla completa es demasiado grande, se usa **búsqueda aleatoria** (Bergstra y Bengio, 2012), que explora mejor los hiperparámetros importantes que una malla del mismo costo; con 30–40 muestras de 480–2,592 combinaciones, el resultado es una configuración buena, no *la* óptima. Cada búsqueda guarda para cada combinación las ocho métricas, la desviación entre folds y el score en entrenamiento (para medir el *gap*).

**Criterio de selección, definido antes de ver los resultados:** el ensamble final es el que tenga mayor F1 macro en CV entre los ensambles optimizados. La selección se hace con CV; Test no interviene.

In [ ]:
METRICA_REFIT = Registro.METRICA_REFIT        # criterio de selección declarado de antemano: F1 macro (igual que en la práctica de SVM)
# buscar() y tabla_busqueda() están en config.py (las búsquedas se registran en `searches` / `tiempos_busqueda`)
print("Métrica de selección:", METRICA_REFIT, "| búsquedas con los mismos 5 folds")

### 17.1 Random Forest
Búsqueda aleatoria de 40 de las 840 combinaciones. Los hiperparámetros que más controlan el sobreajuste de un bosque son `min_samples_leaf` y `max_depth` (tamaño de cada árbol) y `max_features` (qué tan decorrelacionados quedan los árboles); `class_weight` decide si el bosque sacrifica Accuracy por reconocer a *Enrolled*.

In [ ]:
buscar("Random Forest", crear_pipeline(RandomForestClassifier(n_estimators=300, n_jobs=1, random_state=RANDOM_STATE)),
       {"model__max_features": ["sqrt", "log2", 0.2, 0.35, 0.5],
        "model__min_samples_leaf": [1, 2, 3, 5, 8, 12, 20],
        "model__max_depth": [None, 8, 12, 16],
        "model__class_weight": [None, "balanced", "balanced_subsample"],
        "model__criterion": ["gini", "entropy"]}, n_iter=40)

print("Random Forest — 8 mejores combinaciones (F1 macro en CV):")
display(tabla_busqueda(searches["Random Forest"], n=8).round(4))

**Interpretación.**
- **Mejor configuración:** `min_samples_leaf = 2`, `max_features = "log2"` (6 de las 106 columnas por división), `max_depth = 16`, `criterion = "entropy"` y `class_weight = "balanced"`, con F1 macro 0.7262 ± 0.0221 en CV. Mejora +0.009 al Random Forest base (0.7176), menos de la mitad de una desviación entre folds.
- **Los ocho mejores están todos entre 0.7119 y 0.7262**: la superficie es plana y el orden entre ellos es en gran parte ruido. Lo que sí es consistente es que **ninguno usa `class_weight = None`** (6 de 8 usan `"balanced"` y 2 `"balanced_subsample"`): ponderar las clases es lo único que se ve claramente en los datos. Con `criterion` hay empate (4 *gini* y 4 *entropy*).
- **El primer lugar es también el de mayor sobreajuste:** gap 0.2025 (F1 en entrenamiento 0.9287), contra 0.11 del segundo (0.7188). El criterio declarado de antemano es el F1 macro, así que se elige el primero, pero con 40 combinaciones de ruido similar, el máximo está sobreestimado (*winner's curse*); la CV repetida de la Fase 7 lo verifica.

### 17.2 Bagging y Pasting
Mismo espacio para los dos (30 combinaciones aleatorias de 480 cada uno); lo único que cambia es si el muestreo es con o sin reemplazo. `max_samples` es el tamaño de la submuestra de cada árbol y `max_features`, la fracción de columnas que ve cada árbol (con 1.0 son *bagging* y *pasting* puros; con menos, *random subspaces*).

In [ ]:
# --- Bagging y Pasting (mismo espacio; lo único que cambia es bootstrap) ---
espacio_bag = {"model__estimator__max_depth": [4, 6, 8, 12, None],
               "model__estimator__min_samples_leaf": [1, 5, 10, 20],
               "model__estimator__class_weight": [None, "balanced"],
               "model__max_features": [0.3, 0.6, 1.0]}
buscar("Bagging", crear_pipeline(BaggingClassifier(estimator=DecisionTreeClassifier(random_state=RANDOM_STATE),
        n_estimators=200, bootstrap=True, n_jobs=1, random_state=RANDOM_STATE)),
       {**espacio_bag, "model__max_samples": [0.3, 0.5, 0.7, 1.0]}, n_iter=30)
buscar("Pasting", crear_pipeline(BaggingClassifier(estimator=DecisionTreeClassifier(random_state=RANDOM_STATE),
        n_estimators=200, bootstrap=False, n_jobs=1, random_state=RANDOM_STATE)),
       {**espacio_bag, "model__max_samples": [0.3, 0.5, 0.7, 0.9]}, n_iter=30)

print("Bagging — 8 mejores combinaciones (F1 macro en CV):")
display(tabla_busqueda(searches["Bagging"], n=8).round(4))

print("Pasting — 8 mejores combinaciones (F1 macro en CV):")
display(tabla_busqueda(searches["Pasting"], n=8).round(4))

**Interpretación.**
- **Bagging:** `max_samples = 1.0`, `max_features = 0.3`, árbol base con `min_samples_leaf = 5`, `max_depth = 12` y `class_weight = "balanced"`: F1 macro 0.7101 ± 0.0195 (base: 0.7023). **Pasting:** `max_samples = 0.9`, mismos hiperparámetros del árbol: 0.7189 ± 0.0200 (base: 0.7004).
- **Los dos eligen `max_features = 0.3`** (cada árbol ve solo ≈32 de las 106 columnas, la técnica de *random subspaces*): decorrelaciona los árboles, igual que `max_features` en un Random Forest. Y casi todos los mejores usan `class_weight = "balanced"` en el árbol base (7 de 8 en *bagging*, 8 de 8 en *pasting*).
- **El *pasting* queda 0.009 por encima del *bagging*** en la búsqueda, pero con desviaciones de ≈0.02 no es una diferencia que se pueda sostener; ambos se quedan por debajo del Random Forest optimizado (0.7262).
- **Hay configuraciones casi iguales con la mitad de sobreajuste**: en *bagging*, el segundo lugar (`max_samples = 0.7`, `max_features = 0.6`, `min_samples_leaf = 10`) tiene F1 macro 0.7099 con un gap de 0.085, contra 0.171 del primero. El mejor F1 y el mejor compromiso no son la misma configuración.

### 17.3 Boosting
- **AdaBoost:** búsqueda exhaustiva de 140 combinaciones (el espacio se amplió después de ver que, con una malla menor, la mejor profundidad quedaba en el borde).
- **Gradient Boosting:** 30 de 1,728 combinaciones. `learning_rate` y `n_estimators` trabajan juntos (pasos pequeños requieren más árboles); `max_depth`, `min_samples_leaf`, `subsample` y `max_features` regularizan cada árbol.
- **HistGradientBoosting:** 40 de 2,592 combinaciones, con `l2_regularization` y `max_leaf_nodes` como controles adicionales.

In [ ]:
# --- AdaBoost ---
buscar("AdaBoost", crear_pipeline(AdaBoostClassifier(
        estimator=DecisionTreeClassifier(class_weight="balanced", random_state=RANDOM_STATE), random_state=RANDOM_STATE)),
       {"model__n_estimators": [25, 50, 100, 200, 400], "model__learning_rate": [0.03, 0.1, 0.3, 1.0],
        "model__estimator__max_depth": [1, 2, 3, 4, 6, 8, 10]})

print("AdaBoost — 8 mejores combinaciones (F1 macro en CV):")
display(tabla_busqueda(searches["AdaBoost"], n=8).round(4))

In [ ]:
# --- Gradient Boosting (balanceado) ---
buscar("Gradient Boosting", crear_pipeline(GradientBoostingBalanceado(random_state=RANDOM_STATE)),
       {"model__learning_rate": [0.02, 0.05, 0.1, 0.2], "model__n_estimators": [100, 200, 300],
        "model__max_depth": [2, 3, 4, 5], "model__subsample": [0.5, 0.7, 1.0],
        "model__min_samples_leaf": [1, 5, 10, 20], "model__max_features": [None, "sqrt", 0.3]}, n_iter=30)

print("Gradient Boosting — 8 mejores combinaciones (F1 macro en CV):")
display(tabla_busqueda(searches["Gradient Boosting"], n=8).round(4))

In [ ]:
# --- HistGradientBoosting ---
buscar("HistGradientBoosting", crear_pipeline(HistGradientBoostingClassifier(early_stopping=False, random_state=RANDOM_STATE)),
       {"model__learning_rate": [0.02, 0.05, 0.1, 0.2], "model__max_iter": [100, 200, 400],
        "model__max_leaf_nodes": [7, 15, 31], "model__min_samples_leaf": [10, 20, 40, 80],
        "model__l2_regularization": [0, 1, 10], "model__max_depth": [None, 3, 5],
        "model__class_weight": [None, "balanced"]}, n_iter=40)

print("HistGradientBoosting — 8 mejores combinaciones (F1 macro en CV):")
display(tabla_busqueda(searches["HistGradientBoosting"], n=8).round(4))

**Interpretación.**
- **AdaBoost:** `max_depth = 6`, `learning_rate = 0.1` y `n_estimators = 50`, con F1 macro 0.7177 ± 0.0187 (base: 0.6784, +0.039): es la mejora más grande de una optimización, y sale de abandonar los árboles de profundidad 2. La profundidad ganadora es interior en la malla ampliada (se exploró hasta 10). **Es también la configuración más regularizada de las mejores:** gap 0.087, contra 0.23–0.28 de las que usan árboles de profundidad 8 a 10 con el mismo F1 aproximado (0.714 a 0.717). Los tres resultados idénticos (0.7152, con `max_depth = 6`, `learning_rate = 0.1` y `n_estimators` 100, 200 y 400) confirman que el algoritmo se detiene solo antes de llegar a esos números de estimadores.
- **Gradient Boosting:** `learning_rate = 0.05`, 300 árboles de profundidad 4, `subsample = 0.7`, `max_features = 0.3` y `min_samples_leaf = 5`: F1 macro 0.7316 ± 0.0191, apenas +0.0016 sobre el modelo base (0.7300) y con más gap (0.185 contra 0.167). Los ocho mejores están entre 0.7261 y 0.7316, y seis usan `learning_rate = 0.05`, con árboles pequeños (profundidad 3 a 5). **Gradient Boosting ya venía bien configurado y la búsqueda no encontró nada claramente mejor.** Hay configuraciones casi iguales con mucho menos sobreajuste: el cuarto lugar (200 árboles, profundidad 3, `max_features = "sqrt"`) tiene 0.7278 con gap 0.058.
- **HistGradientBoosting:** `max_leaf_nodes = 15`, `min_samples_leaf = 10`, `max_iter = 100`, `learning_rate = 0.1`, `l2_regularization = 1`, sin límite de profundidad y `class_weight = "balanced"`: F1 macro 0.7340 ± 0.0189 (base: 0.7164, +0.018) y gap de 0.213 (base: 0.282). Las siete primeras combinaciones usan `class_weight = "balanced"`. Aquí **sí hay una mejora**, que viene de entrenar la mitad de iteraciones (100 contra 200) y regularizar con L2. Tres de sus hiperparámetros (`max_iter`, `min_samples_leaf` y `max_depth`) quedaron en el borde de su rango; se revisa en la Sección 19.

### 17.4 Explainable Boosting Machine
Búsqueda exhaustiva de 24 combinaciones: número de interacciones (0 es el GAM puro), `learning_rate`, `max_leaves` (profundidad de cada árbol de un solo paso) y `min_samples_leaf`. Es la búsqueda más lenta por costo por ajuste, no por número de combinaciones.

In [ ]:
# --- EBM (GAM con boosting) ---
with warnings.catch_warnings():
    warnings.simplefilter("ignore", UserWarning)
    buscar("EBM", crear_ebm(0),
           {"model__interactions": [0, 10, 20], "model__learning_rate": [0.01, 0.03],
            "model__max_leaves": [2, 3], "model__min_samples_leaf": [4, 20]})

print("EBM — 10 mejores combinaciones (F1 macro en CV):")
display(tabla_busqueda(searches["EBM"], n=10).round(4))

**Interpretación.**

- **Mejor configuración:** `interactions = 10`, `learning_rate = 0.03`, `max_leaves = 3` y `min_samples_leaf = 4`, con F1 macro 0.7309 ± 0.0139. Frente al EBM base con 10 interacciones (0.7285) la mejora es de +0.0024, es decir, **prácticamente nula**. Frente al GAM sin interacciones (0.7143) sí hay +0.017, pero esa diferencia viene de *tener* interacciones, no de ajustar los demás hiperparámetros.
- **Las diez mejores combinaciones están en un intervalo de 0.0061** (0.7248 a 0.7309), menos de la mitad de la desviación estándar entre folds (≈ 0.014). Elegir entre ellas es elegir entre empates. Todas tienen interacciones (10 o 20), así que `interactions = 0` queda por debajo de 0.7248; entre 10 y 20 no hay diferencia apreciable (0.7309 contra 0.7297), y lo mismo pasa con `min_samples_leaf` 4 contra 20 (0.7309 contra 0.7297).
- **Es el modelo con menos sobreajuste de la tabla:** gap train − CV de 0.064, contra 0.17–0.21 de Random Forest, Bagging, Pasting, Gradient Boosting e HistGradientBoosting y 0.087 de AdaBoost. Es el costo de ser un modelo aditivo: no puede memorizar combinaciones raras de variables, solo las interacciones que se le permiten.
- **Es también el de mayor *recall* macro** entre los optimizados (0.7398, apenas por encima de *Voting hard* y Stacking, 0.7395 y 0.7387), lo que va con su mejor desempeño en la clase *Enrolled* (ver la Sección 22).
- **Es la búsqueda con mayor costo por ajuste:** 24 combinaciones × 5 folds tomaron 43 s, contra 11 a 18 s de las demás búsquedas (salvo AdaBoost, 35 s, que explora 140 combinaciones). Un EBM tarda ≈ 3 s en ajustarse, de 2 a 6 veces más que Random Forest, Bagging, Pasting, AdaBoost, Gradient Boosting o HistGradientBoosting (0.5 a 1.6 s).

### 18. Mapas de calor y curvas de los hiperparámetros
**Random Forest: tamaño de las hojas contra número de variables por división.** Se fijan los demás hiperparámetros en los de la mejor configuración y se barre una malla de 6 × 6 de `min_samples_leaf` × `max_features`, mostrando el F1 macro en CV, el F1 en entrenamiento y su diferencia (el *gap*). Después se mide el efecto del **número de árboles**: un bosque no sobreajusta por tener más árboles; su error se estabiliza y más árboles solo cuestan tiempo.

In [ ]:
# Mapa de calor del Random Forest: min_samples_leaf × max_features (el resto de hiperparámetros, en su mejor valor)
rf_mejor = searches["Random Forest"].best_estimator_
espacio_rf = {"model__min_samples_leaf": [1, 2, 5, 10, 20, 40],
              "model__max_features": [0.05, 0.1, 0.2, 0.3, 0.5, 1.0]}
buscar("Random Forest (mapa de calor)", clone(rf_mejor), espacio_rf)

res_rf = pd.DataFrame(searches["Random Forest (mapa de calor)"].cv_results_)
res_rf["min_samples_leaf"] = res_rf["param_model__min_samples_leaf"].astype(int)
res_rf["max_features"] = res_rf["param_model__max_features"].astype(float)
fig, axes = plt.subplots(1, 3, figsize=(17, 4.6))
for ax, (col, titulo, cmap) in zip(axes, [("mean_test_F1", "F1 macro en CV", "viridis"),
                                           ("mean_train_F1", "F1 macro en train", "viridis"),
                                           ("gap", "Gap (train − CV)", "magma_r")]):
    res_rf["gap"] = res_rf["mean_train_F1"] - res_rf["mean_test_F1"]
    sns.heatmap(res_rf.pivot(index="min_samples_leaf", columns="max_features", values=col), annot=True, fmt=".3f",
                cmap=cmap, ax=ax, cbar=False)
    ax.set_title(f"Random Forest — {titulo}")
    ax.set_xlabel("max_features (fracción de las 106 columnas)")
    ax.set_ylabel("min_samples_leaf")
plt.tight_layout()
guardar_fig("03_rf_mapa_calor")

# Número de árboles: el error de un bosque se estabiliza; más árboles no sobreajustan, solo cuestan tiempo
n_arboles = N_ARBOLES
tr, va = validation_curve(clone(rf_mejor), X_train, y_train, param_name="model__n_estimators", param_range=n_arboles,
                          cv=cv, scoring="f1_macro", n_jobs=N_JOBS)
curva_arboles = pd.DataFrame({"F1 train": tr.mean(axis=1), "F1 CV": va.mean(axis=1), "std CV": va.std(axis=1)},
                             index=pd.Index(n_arboles, name="n_estimators"))
fig, ax = plt.subplots(figsize=(7.5, 4.3))
ax.plot(n_arboles, curva_arboles["F1 train"], "o--", label="train")
ax.errorbar(n_arboles, curva_arboles["F1 CV"], yerr=curva_arboles["std CV"], fmt="o-", capsize=3, label="CV (media ± std)")
ax.set_xscale("log")
ax.set_xlabel("Número de árboles")
ax.set_ylabel("F1 macro")
ax.set_title("Random Forest — F1 macro según el número de árboles")
ax.legend()
plt.tight_layout()
guardar_fig("03_rf_numero_de_arboles")
curva_arboles.round(4)

**Interpretación.**

- **`min_samples_leaf` es el hiperparámetro que domina el mapa; `max_features` casi no importa.** En cada columna, el F1 de CV baja de ≈ 0.72 con hojas de 1 a 2 observaciones a ≈ 0.69 con hojas de 40. En cada fila, en cambio, moverse entre `max_features` 0.05 y 1.0 cambia el F1 en menos de 0.02, con un máximo suave en 0.1 a 0.3 (el mejor es 0.2 con hojas de 1: 0.725, gap 0.274).
- **El mejor F1 y el menor sobreajuste están en esquinas opuestas.** Con hojas de 1 el F1 de entrenamiento es ≈ 1.00 (gap 0.27 a 0.29); con hojas de 40 el gap es de 0.02 a 0.03, pero el F1 de CV cae unos 0.035. Entre `min_samples_leaf` 1 y 5 la diferencia de F1 (0.725 contra 0.719 con `max_features = 0.2`) es menor que una desviación estándar (≈ 0.02), mientras que el gap baja de 0.274 a 0.159. **Estadísticamente son empates; en términos de sobreajuste, no.** La búsqueda de la Sección 17.1 eligió hojas de 2 por el criterio fijado de antemano (F1 de CV), pero habría sido igual de defendible quedarse con 5.
- **Más árboles no sobreajustan, pero tampoco ayudan.** El F1 de CV sube de 0.690 con 10 árboles a 0.715 con 25 y 0.722 con 50, y desde 100 árboles se queda en 0.722 a 0.726 sin tendencia (la desviación estándar entre folds es de 0.02 a 0.027). El F1 de entrenamiento también se estabiliza (0.927 a 0.928). Pasar de 100 a 800 árboles multiplica el costo por 8 sin ganancia medible; con 100 a 200 es suficiente.

**Boosting: el compromiso entre `learning_rate` y número de árboles.** En *boosting*, sí se sobreajusta al añadir árboles. Se entrena cada modelo con la configuración óptima pero con muchos árboles y se mide el F1 macro tras cada árbol añadido (con `staged_predict`) en los 5 folds, para varios `learning_rate`: la curva de validación sube, alcanza un máximo y luego baja o se aplana, mientras la de entrenamiento sigue creciendo.

In [ ]:
# Boosting: F1 macro tras cada árbol añadido, para varios learning rates (CV de 5 folds, una sola corrida por learning rate)
import warnings; warnings.filterwarnings("ignore", message="A worker stopped")
from joblib import Parallel, delayed

def una_corrida(clave, param_n, lr, n_max, tr, va):
    pipe = clone(searches[clave].best_estimator_).set_params(model__learning_rate=lr, **{f"model__{param_n}": n_max})
    pipe.fit(X_train.iloc[tr], y_train[tr])
    prep, modelo = pipe.named_steps["preprocessor"], pipe.named_steps["model"]
    f_tr = [f1_score(y_train[tr], p, average="macro") for p in modelo.staged_predict(prep.transform(X_train.iloc[tr]))]
    f_va = [f1_score(y_train[va], p, average="macro") for p in modelo.staged_predict(prep.transform(X_train.iloc[va]))]
    return f_tr, f_va

def curva_etapas(clave, param_n, lrs, n_max):
    folds = list(cv.split(X_train, y_train))
    salida = Parallel(n_jobs=N_JOBS)(delayed(una_corrida)(clave, param_n, lr, n_max, tr, va)
                                     for lr in lrs for tr, va in folds)
    out = {}
    for i, lr in enumerate(lrs):
        parte = salida[i * len(folds):(i + 1) * len(folds)]
        out[lr] = (np.mean([p[0] for p in parte], axis=0), np.mean([p[1] for p in parte], axis=0),
                   np.std([p[1] for p in parte], axis=0))
    return out

curvas_boost = {"Gradient Boosting": curva_etapas("Gradient Boosting", "n_estimators", [0.02, 0.05, 0.1, 0.2], N_MAX_GB),
                "HistGradientBoosting": curva_etapas("HistGradientBoosting", "max_iter", [0.03, 0.1, 0.3], N_MAX_HGB)}

fig, axes = plt.subplots(1, 2, figsize=(15, 4.8))
for ax, (nombre, curvas) in zip(axes, curvas_boost.items()):
    for (lr, (f_tr, f_va, s_va)), color in zip(curvas.items(), sns.color_palette("viridis", len(curvas))):
        x = np.arange(1, len(f_va) + 1)
        ax.plot(x, f_va, color=color, label=f"CV, lr={lr}")
        ax.plot(x, f_tr, color=color, ls=":", lw=1)
        ax.plot(x[f_va.argmax()], f_va.max(), "o", color=color)
    ax.set_xlabel("Árboles (iteraciones) añadidos")
    ax.set_ylabel("F1 macro")
    ax.set_title(f"{nombre} (línea: CV; punteada: train; punto: mejor CV)")
    ax.legend(loc="lower right", fontsize=8.5)
plt.tight_layout()
guardar_fig("03_boosting_curvas_por_etapa")

filas = []
for nombre, curvas in curvas_boost.items():
    for lr, (f_tr, f_va, s_va) in curvas.items():
        filas.append({"Modelo": nombre, "learning_rate": lr, "Mejor nº de árboles": int(f_va.argmax()) + 1,
                      "F1 CV en el mejor": f_va.max(), "F1 CV al final": f_va[-1], "Gap al final (train−CV)": f_tr[-1] - f_va[-1]})
pd.DataFrame(filas).set_index(["Modelo", "learning_rate"]).round(4)

**Interpretación.**

- **El techo es el mismo para cualquier `learning_rate`.** El mejor F1 de CV de cada curva está entre 0.727 y 0.735 en Gradient Boosting y entre 0.735 y 0.737 en HistGradientBoosting. Esas diferencias (< 0.01) son menores que la desviación estándar entre folds (≈ 0.019). Lo que cambia con la tasa de aprendizaje es *cuántos árboles* hacen falta: 434 con 0.02, 262 con 0.05, 150 con 0.1 y 42 con 0.2 en Gradient Boosting; 155, 94 y 22 con 0.03, 0.1 y 0.3 en HistGradientBoosting. Las tasas pequeñas llegan al mismo lugar con un camino más largo.
- **Pasarse de árboles cuesta, y cuesta más con tasas grandes.** Al final de la curva el F1 de CV es menor que en el mejor punto: −0.0025 con `learning_rate = 0.02`, pero −0.018 con 0.2 en Gradient Boosting y −0.022 con 0.3 en HistGradientBoosting. Mientras tanto el F1 de entrenamiento llega a ≈ 1.00 y el gap final de Gradient Boosting es de 0.14 con la tasa más baja y de 0.29 con la más alta. **Una tasa pequeña perdona más un `n_estimators` mal elegido.**
- **Estas curvas muestran el máximo de una curva ruidosa, así que son optimistas.** Los máximos de las curvas (hasta 0.7366) están un poco por encima de lo que encontró la búsqueda de la Sección 17.3 (0.7316 y 0.7340), porque aquí se elige el mejor punto entre 300 a 500 posibles *sobre los mismos folds* con los que se mide. Sirven para entender la forma de la curva, no para reportar un F1.

### 19. ¿Los óptimos caen en el borde de su rango?
Si el mejor valor de un hiperparámetro numérico es el mínimo o el máximo de lo explorado, el óptimo real podría estar fuera de la malla. La tabla lo revisa automáticamente para todas las búsquedas.

In [ ]:
# ¿El mejor valor de cada hiperparámetro numérico cae en el borde de su rango? Si sí, el óptimo podría estar fuera de lo explorado.
def es_num(v):
    return isinstance(v, (int, float, np.integer, np.floating)) and not isinstance(v, bool)

filas = []
for nombre, s in searches.items():
    if nombre.startswith("Voting") or "mapa de calor" in nombre:
        continue
    espacio = s.param_grid if hasattr(s, "param_grid") else s.param_distributions
    for p, rango in espacio.items():
        mejor = s.best_params_[p]
        nums = [v for v in rango if es_num(v)]
        if mejor is None:
            borde = "sí (sin límite)"
        elif es_num(mejor) and len(nums) >= 3:
            borde = "sí" if mejor in (min(nums), max(nums)) else "no"
        else:
            borde = "—"                                   # categórico o con solo 2 valores
        filas.append({"Modelo": nombre, "Hiperparámetro": p.replace("model__", ""),
                      "Rango explorado": ", ".join("None" if v is None else str(v) for v in rango),
                      "Mejor": "None" if mejor is None else mejor, "¿En el borde?": borde})
tabla_borde = pd.DataFrame(filas)
display(tabla_borde.style.hide(axis="index"))
print("Hiperparámetros numéricos en el borde, por modelo:")
display(tabla_borde[tabla_borde["¿En el borde?"].str.startswith("sí")].groupby("Modelo")["Hiperparámetro"].apply(", ".join).to_frame())

**Interpretación.**

La tabla marca como "borde" a todo valor numérico igual al mínimo o al máximo explorado. Hay que leerla con cuidado, porque no todas las marcas son lo mismo:

- **Falsos bordes.** `max_depth = 16` en Random Forest y `max_depth = 12` en Bagging y Pasting son el mayor valor *finito* de su rango, pero ese rango incluye también `None` (sin límite), que se probó y no ganó: la profundidad no está truncada por la malla. En HistGradientBoosting gana `max_depth = None`, es decir, no hace falta limitar la profundidad porque `max_leaf_nodes = 15` ya acota el árbol (y ese sí es un valor interior).
- **Bordes que pueden importar.** `max_features = 0.3` en Bagging y Pasting es el valor más bajo explorado. Es consistente con el mapa de calor del Random Forest (Sección 18), donde 0.1 a 0.3 funcionaba mejor que 0.5 a 1.0, así que es posible que valores de 0.1 a 0.2 den una pizca más. `max_samples = 1.0` (Bagging) y `0.9` (Pasting) son los más altos explorados; en Bagging 1.0 es el valor natural (el *bootstrap* ya deja fuera ≈ 37 % de los datos en cada árbol).
- **Bordes en los boosting.** Gradient Boosting eligió `n_estimators = 300`, el máximo de 100, 200 y 300, pero la curva por etapas de la Sección 18 muestra que con `learning_rate = 0.05` el óptimo está en ≈ 262 árboles y que el F1 de CV baja después: el óptimo no está más allá del borde. HistGradientBoosting eligió `max_iter = 100` y `min_samples_leaf = 10`, ambos los valores más bajos probados; con `learning_rate = 0.1` la curva por etapas tiene su mejor punto en 94 iteraciones y es plana alrededor (0.7352 contra 0.7340 en 100), y hojas más chicas podrían ganar un poco, pero no se espera una diferencia mayor que la que ya hay entre combinaciones vecinas (≤ 0.01).
- **Decisión:** no se amplían las mallas. Las ganancias posibles son del tamaño del ruido entre folds (≈ 0.02) y seguir buscando sobre los mismos folds aumentaría el sesgo optimista de la selección (la *maldición del ganador*, Sección 23). Queda señalado como limitación.

### 20. Evaluación de los modelos optimizados
Cada configuración ganadora se reentrena y se evalúa con el procedimiento completo de la Fase 5 (CV de 5 folds, predicciones OOF, ajuste final con todo Train). Para el *Voting* se eligen los pesos con una búsqueda sobre 5 vectores; para el *Stacking* el *blender* es una `LogisticRegressionCV` que escoge su $C$ con una validación cruzada interna sobre sus 15 columnas (5 modelos × 3 clases), sin volver a ajustar los modelos base por cada $C$. Los miembros de *voting* y *stacking* son ahora cinco modelos de distinta naturaleza: regresión logística, SVM RBF, y los Random Forest, Gradient Boosting e HistGradientBoosting **ya optimizados**.

In [ ]:
# Cada modelo optimizado se reentrena con los mejores hiperparámetros de su búsqueda (mismos folds de CV)
modelos_opt = {"Random Forest": "Random Forest optimizado", "Bagging": "Bagging optimizado",
               "Pasting": "Pasting optimizado", "AdaBoost": "AdaBoost optimizado",
               "Gradient Boosting": "Gradient Boosting optimizado",
               "HistGradientBoosting": "HistGradientBoosting optimizado", "EBM": "EBM optimizado"}
with warnings.catch_warnings():
    warnings.simplefilter("ignore", UserWarning)
    for clave, nombre in modelos_opt.items():
        evaluar_cv(nombre, clone(searches[clave].best_estimator_))

In [ ]:
def interno(clave):
    """Modelo (sin preprocesador) de la mejor configuración de una búsqueda."""
    return clone(searches[clave].best_estimator_.named_steps["model"])

def miembros_opt(probabilidades=False):
    """Cinco miembros de naturaleza distinta, todos con sus mejores hiperparámetros."""
    lr = LogisticRegression(**BEST_LR, max_iter=5000, random_state=RANDOM_STATE)
    svm = SVC(**BEST_SVM["SVM RBF optimizado"]["params"])
    if probabilidades:
        svm = CalibratedClassifierCV(svm, method="sigmoid", cv=5, ensemble=False)
    return [("lr", lr), ("svm", svm), ("rf", interno("Random Forest")),
            ("gb", interno("Gradient Boosting")), ("hgb", interno("HistGradientBoosting"))]

pesos_voting = {"uniforme (1,1,1,1,1)": [1, 1, 1, 1, 1], "árboles ×2 (1,1,2,2,2)": [1, 1, 2, 2, 2],
                "LR ×2 (2,1,1,1,1)": [2, 1, 1, 1, 1], "boosting ×2 (1,1,1,2,2)": [1, 1, 1, 2, 2],
                "sin SVM (1,0,1,1,1)": [1, 0, 1, 1, 1]}
for tipo in ["hard", "soft"]:
    buscar(f"Voting ({tipo})", crear_pipeline(VotingClassifier(miembros_opt(tipo == "soft"), voting=tipo)),
           {"model__weights": list(pesos_voting.values())}, sin_auc=(tipo == "hard"))
    print(f"Voting ({tipo}) — pesos (LR, SVM, RF, GB, HGB) ordenados por F1 macro en CV:")
    display(tabla_busqueda(searches[f"Voting ({tipo})"], n=5).round(4))
    evaluar_cv(f"Voting ({tipo}) optimizado", clone(searches[f"Voting ({tipo})"].best_estimator_))

# Stacking: el meta-modelo (regresión logística balanceada) elige su propio C con una validación cruzada interna
# sobre sus 15 columnas (5 modelos × 3 clases), que es barata; los modelos base no se vuelven a ajustar por cada C.
blender = LogisticRegressionCV(Cs=[0.01, 0.1, 1, 10], l1_ratios=(0.0,), cv=5, scoring="f1_macro",
                               class_weight="balanced", max_iter=2000, use_legacy_attributes=False,
                               random_state=RANDOM_STATE)
stacking_opt = crear_pipeline(StackingClassifier(
    estimators=miembros_opt(False), final_estimator=blender,
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE), stack_method="auto", n_jobs=1))
_ = evaluar_cv("Stacking optimizado", stacking_opt)

### 21. Efecto de la optimización
Comparación de cada modelo con hiperparámetros base contra su versión optimizada (CV de 5 folds), con el *gap* train − CV de ambas.

In [ ]:
pares = [("Random Forest", "Random Forest optimizado"), ("Bagging (árboles)", "Bagging optimizado"),
         ("Pasting (árboles)", "Pasting optimizado"), ("AdaBoost", "AdaBoost optimizado"),
         ("Gradient Boosting (balanceado)", "Gradient Boosting optimizado"),
         ("HistGradientBoosting (balanceado)", "HistGradientBoosting optimizado"), ("EBM (10 interacciones)", "EBM optimizado"),
         ("Voting (hard)", "Voting (hard) optimizado"), ("Voting (soft)", "Voting (soft) optimizado"),
         ("Stacking (LR + SVM + RF → LR)", "Stacking optimizado")]
comp = pd.DataFrame({o.replace(" optimizado", ""): {
    "F1 base": resultados_cv[b]["F1_mean"], "F1 optimizado": resultados_cv[o]["F1_mean"],
    "Δ F1": resultados_cv[o]["F1_mean"] - resultados_cv[b]["F1_mean"],
    "std optimizado": resultados_cv[o]["F1_std"],
    "Gap base": resultados_cv[b]["gap_F1"], "Gap optimizado": resultados_cv[o]["gap_F1"],
    "Accuracy base → opt.": f"{resultados_cv[b]['Accuracy_mean']:.4f} → {resultados_cv[o]['Accuracy_mean']:.4f}",
    "MCC base → opt.": f"{resultados_cv[b]['MCC_mean']:.4f} → {resultados_cv[o]['MCC_mean']:.4f}",
    "ROC-AUC base → opt.": ("—" if np.isnan(resultados_cv[o]["ROC-AUC_mean"]) else
                            f"{resultados_cv[b]['ROC-AUC_mean']:.4f} → {resultados_cv[o]['ROC-AUC_mean']:.4f}")}
    for b, o in pares}).T
display(comp)
print(f"Referencia: {NOMBRE_LR} F1 macro en CV = {resultados_cv[NOMBRE_LR]['F1_mean']:.4f} ± {resultados_cv[NOMBRE_LR]['F1_std']:.4f}")
print("Resumen completo de los modelos optimizados (CV, 5 folds):")
display(tabla_resumen([NOMBRE_LR, "SVM RBF optimizado"] + [o for _, o in pares]))

fig, ax = plt.subplots(figsize=(10, 5.5))
y_pos = np.arange(len(pares))[::-1]
for yp, (b, o) in zip(y_pos, pares):
    ax.plot([resultados_cv[b]["F1_mean"], resultados_cv[o]["F1_mean"]], [yp, yp], color=COLOR_GRIS, lw=1.5, zorder=1)
ax.scatter([resultados_cv[b]["F1_mean"] for b, _ in pares], y_pos, color="tab:gray", s=45, label="hiperparámetros base", zorder=2)
ax.scatter([resultados_cv[o]["F1_mean"] for _, o in pares], y_pos, color="tab:blue", s=55, label="optimizado", zorder=3)
ax.axvline(resultados_cv[NOMBRE_LR]["F1_mean"], color="tab:red", ls="--", lw=1.2, label="regresión logística (F_1)")
ax.set_yticks(y_pos)
ax.set_yticklabels([o.replace(" optimizado", "") for _, o in pares])
ax.set_xlabel("F1 macro en CV (5 folds)")
ax.set_title("Efecto de la optimización de hiperparámetros")
ax.legend(loc="lower right")
plt.tight_layout()
guardar_fig("03_antes_vs_despues_optimizacion")
guardar_tabla(comp, "03_optimizacion_base_vs_optimizado")
guardar_tabla(tabla_resumen([NOMBRE_LR, "SVM RBF optimizado"] + [o for _, o in pares]), "03_modelos_optimizados_cv")

**Interpretación.**

- **La optimización mejora el F1 de CV de los 10 ensambles, pero poco:** +0.0127 en promedio, de +0.0016 (Gradient Boosting) a +0.0392 (AdaBoost), contra una desviación estándar entre folds de 0.014 a 0.022. Solo cinco ganan más de 0.01: AdaBoost (+0.039), Pasting (+0.018), HistGradientBoosting (+0.018), *Voting soft* (+0.012) y Stacking (+0.011). Gradient Boosting y el EBM ya venían bien configurados (+0.0016 y +0.0024): ahí no hay ganancia medible.
- **Casi todo lo que se gana en F1 se paga en sobreajuste.** El gap train − CV sube en 9 de los 10 modelos (en promedio de 0.099 a 0.163); la única excepción es HistGradientBoosting (0.282 a 0.213). Bagging pasó de 0.052 a 0.171, Pasting de 0.065 a 0.193 y *Voting soft* de 0.052 a 0.178. Buscar el mejor F1 de CV empuja hacia modelos más flexibles (árboles más profundos, menos regularización), que ganan unas milésimas en validación y se acercan a memorizar Train.
- **La Accuracy no es lo que se optimizó, y en los *voting* incluso baja** (*soft*: 0.7898 a 0.7858; *hard*: 0.7796 a 0.7793), mientras el *recall* macro del *soft* sube de 0.7149 a 0.7347. El ROC-AUC casi no cambia (+0.001 a +0.007, salvo AdaBoost, +0.029): la búsqueda mueve el punto de operación, no la calidad del ordenamiento.
- **Después de optimizar los ensambles quedan en 0.7101 a 0.7375 de F1 macro; los seis primeros, en un rango de 0.0066** (0.7309 a 0.7375), mucho menos que una desviación estándar. Por el criterio declarado de antemano gana *Voting soft*, pero supera a *Voting hard* por 0.0002. **Que sea el "mejor" es una decisión de procedimiento, no una evidencia de superioridad.** La regresión logística de referencia tiene 0.7170 ± 0.0192.

### Guardado para el notebook 04
Se guardan los modelos ajustados con todo Train, las predicciones *out-of-fold*, las métricas de CV y las búsquedas en `outputs/modelos/registro.joblib`,
y los mejores hiperparámetros de cada búsqueda en `outputs/tablas/03_mejores_hiperparametros.json`.

In [ ]:
reg.guardar()

mejores = {n: {k.replace("model__", ""): v for k, v in s.best_params_.items()}
           for n, s in searches.items() if "mapa de calor" not in n}
with open(TABLAS / "03_mejores_hiperparametros.json", "w", encoding="utf-8") as f:
    json.dump(mejores, f, indent=2, ensure_ascii=False, default=lambda o: o.item() if hasattr(o, "item") else str(o))

print(f"{len(modelos_ajustados)} modelos y {len(searches)} búsquedas guardados en outputs/modelos/registro.joblib")
print("Mejores hiperparámetros → outputs/tablas/03_mejores_hiperparametros.json")